#主成分分析により多重共線性の解消を目指す

In [ ]:
#import
import pandas as pd
from sklearn.model_selection import KFold #K分割交差検証
from sklearn.model_selection import cross_validate  #K分割交差検証

from sklearn.linear_model import LinearRegression #回帰
from sklearn.preprocessing import PolynomialFeatures  #交互作用特徴量
from sklearn.linear_model import Ridge  #リッジ回帰
from sklearn.linear_model import Lasso  #ラッソ回帰
from sklearn.tree import DecisionTreeRegressor  #回帰木

from statsmodels.stats.outliers_influence import variance_inflation_factor  #VIF
from sklearn.decomposition import PCA  #PCA

In [51]:
#データフレームの読み込み
'''
df1 VIFの高さを解消するために一部の列を消去する前のデータフレーム
df2 〃した後のデータフレーム

以降は基本的にdf1を用いて分析を行う。
sc_x,df_yはdf1を基に作成する。
ただし、必要があればdf1も利用する。
'''
df1 = pd.read_csv('datafiles/df1_all_col.csv')

sc_x = df1.drop(['SalePrice'], axis = 1)
df_y = pd.DataFrame(df1['SalePrice'])

In [52]:
#K分割交差検証のための準備
kf=KFold(n_splits=5, shuffle=True, random_state=0)

In [ ]:
#モデルの作成
#リッジ回帰
model2 = Ridge(alpha = 100)
model2.fit(sc_x, df_y)

,criterion,'squared_error'
,splitter,'best'
,max_depth,10
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,None
,random_state,0
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,ccp_alpha,0.0


In [54]:
#vifを見る
vif_df = pd.DataFrame()
vif_df["VIF_Factor"] = [variance_inflation_factor(sc_x, i) for i in range(sc_x.shape[1])]
vif_df["features"]=sc_x.columns
df_high_vif = vif_df.loc[vif_df['VIF_Factor'] >= 10]
display(df_high_vif.sort_values('VIF_Factor', ascending=False))

c:\Users\natsu\anaconda3\Lib\site-packages\statsmodels\stats\outliers_influence.py:197: RuntimeWarning: divide by zero encountered in scalar divide
  vif = 1. / (1. - r_squared_i)


,VIF_Factor,features
15,inf,GrLivArea
176,inf,Exterior2nd_CBlock
8,inf,BsmtFinSF1
9,inf,BsmtFinSF2
10,inf,BsmtUnfSF
...,...,...
55,11.641993,Neighborhood_Edwards
187,11.246911,Exterior2nd_Wd Shng
141,10.947886,KitchenQual_TA
136,10.738728,Heating_Grav


#GarageFinish列でまず実験

In [55]:
#GarageFinish列をダミー変数化した列一覧をリスト化し、主成分分析により一列化
GarageFinish_cols = []
for c in sc_x.columns:
    if 'GarageFinish_' in c:
        GarageFinish_cols.append(c)
print(GarageFinish_cols)


['GarageFinish_NA', 'GarageFinish_RFn', 'GarageFinish_Unf']


In [56]:
#累積寄与率の閾値を0.8として、n_componentsを設定
#適切なPCAのための特徴量数の設定
PCAmodel = PCA(whiten = True)
GarageFinish_df = pd.DataFrame()
for c in GarageFinish_cols:
    GarageFinish_df = pd.concat([GarageFinish_df, sc_x[c]], axis = 1)
PCAmodel.fit(GarageFinish_df)
GarageFinish = PCAmodel.transform(sc_x[GarageFinish_cols])

thred = 0.8
final_num = 0
ratio =PCAmodel.explained_variance_ratio_
array = []
for i in range(len(ratio)):
    ruiseki = sum(ratio[0:i+1])    #i+1個めの特徴量までの累積寄与率
    if ruiseki > thred:   #i+1個めの特徴量において初めて累積寄与率がthredを超えるならば
          final_num = i
          break
print(f'PCAにより作成された特徴量数＝{i}')

PCAにより作成された特徴量数＝1


In [58]:
#最適な特徴量数で、主成分分析の実施
PCAmodel = PCA(n_components = final_num, whiten = True)
PCAmodel.fit(GarageFinish_df)

#主成分によるデータフレームをGarageFinish_PCA_dfとする
GarageFinish_PCA = PCAmodel.transform(GarageFinish_df)
GarageFinish_PCA_df = pd.DataFrame(GarageFinish_PCA)

col_name = []
for i in range(final_num):
    name = 'col_name_' + str(i)
    col_name.append(name)
GarageFinish_PCA_df.columns = col_name

In [59]:
#主成分分析により作成した列を、実施前の列と置き換える
for c in sc_x[GarageFinish_cols]:
    sc_x = sc_x.drop([c], axis = 1)
sc_x = pd.concat([sc_x, GarageFinish_PCA_df], axis = 1)

In [ ]:
#PCAによる効果の表示
model2.fit(sc_x, df_y)
result = cross_validate(model2, sc_x, df_y, cv = kf, scoring = 'r2' , return_train_score = True)
print(f'完成したmodel2のスコア＝{sum(result['test_score'])/len(result['test_score'])}')

完成したmodel2のスコア＝0.7777402255226361


In [ ]:
#多重共線性解消前のmodel2でのスコアは　0.7774453479883086 であるから、わずかに精度が向上した。